In [7]:
# Optional: auto-reload project modules on change (skip if autoreload fails, e.g. on Windows)
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
from gbp.shared.dataloader_mock import DataLoaderMock
from gbp.shared.dataloader_graph import DataLoaderGraph
from gbp.rebalancer.dataloader import DataLoaderRebalancer
from gbp.rebalancer.pipeline import Rebalancer

In [9]:
import numpy as np
import pandas as pd

In [10]:
np.random.seed(42)
CONFIG = {
    'n': 20,
    'n_depots': 2,
    'min_threshold': 0.2,
    'max_threshold': 0.8,
    'resource_capacity': 100,
    'num_resources': 3,
    'time_limit_seconds': 10,
    'n_timestamps': 168,
    'start_date': '2025-01-01',
    'inventory_node_type': 'station',
    'depot_node_type': 'depot',
}

# DataLoaderMock  → temporal raw DataFrames (stations, depots, resources, inventory_ts)
# DataLoaderGraph → GraphData (nodes, edges, resources, commodities, inventory per date)
# DataLoaderRebalancer → demand calc + PDP model extracted from a GraphData snapshot
# Rebalancer → solve + postprocess

dataloader_mock = DataLoaderMock(CONFIG)
dataloader_graph = DataLoaderGraph(dataloader_mock, CONFIG)
dataloader_graph.load_data()

dataloader_rebalancer = DataLoaderRebalancer(dataloader_graph, CONFIG)
rebalancer = Rebalancer(dataloader_rebalancer, CONFIG)
rebalancer.run(date=pd.Timestamp('2025-01-03 12:00'))

Solution found — objective: 42756, total distance: 42756, dropped nodes: []


In [11]:
# Inspect a GraphData snapshot
snapshot = dataloader_graph.get_snapshot(pd.Timestamp('2025-01-03 12:00'))
print(snapshot)
print()
print("Available dates:", dataloader_graph.available_dates[[0, -1]])
print("Inventory time-series shape:", dataloader_graph.inventory_timeseries.shape)
print()

# Run rebalancer for a different date
rebalancer2 = Rebalancer(DataLoaderRebalancer(dataloader_graph, CONFIG), CONFIG)
rebalancer2.run(date=pd.Timestamp('2025-01-05 08:00'))

GraphData(nodes=22, edges=462, resources=3, commodities=1, demands=0, inventory=20)

Available dates: DatetimeIndex(['2025-01-01 00:00:00', '2025-01-07 23:00:00'], dtype='datetime64[ns]', freq=None)
Inventory time-series shape: (168, 20)

Solution found — objective: 60828, total distance: 60828, dropped nodes: []


In [ ]:
# Расчёт матрицы дистанций должен быть в отдельной папке или файле. Лучше для начала просто в shared перенести, osrm.py.
# А в ребалансировщике просто его вызывать.

# Далее мне нужно провеорить что использую ресурсы

# МНЕ НУЖНО ОБЯЗАТЕЛЬНО ДОБАВИТЬ ЛОГИКУ КОСТОВ!!!
# Далее нужно добавить контракты
# Далее нужно сделать md для контрактов
# Далее нужно добавить трейсинг
# Далее нужно добавить тесты
# Далее нужно посмотреть что там с ограничениями, добавить пару интересных
# Далее нужно добавить md с описанием ограничений
# Далее нужно сделать диаграммы для ребалансировщика
# Далее нужно сделать главный md ребалансировщика

# Потом уже по аналогии нужно сделать симулятор

# Потом по аналогии нужно сделать планировщик (без мл пока)

# 